# Patient

## Initialization

In [0]:
import sys

sys.path.append('/Workspace/Users/mommensabry@gmail.com/Medical_Insurance_Analytics/lakehouse/libs')
from pyspark.sql.functions import (
    col,
    trim,
    lower,
    upper,
    length,
    substring,
    count,
    min,
    max,
    when,
    lit,
    coalesce,
    row_number,
    current_timestamp
)

from pyspark.sql.window import Window
from silver_utils import *

## Read Bronze table

In [0]:
df = spark.table("medical_insurance.default.patient")

display(df)

## Data Profiling

In [0]:
df.printSchema()

In [0]:
display(df.describe())

In [0]:
from pyspark.sql.functions import col, count, min, max

results = []

for column in df.columns:
    
    # Count unique values
    unique_count = df.select(column).distinct().count()
    
    # Most frequent value and its count
    top_value = (
        df.groupBy(column)
        .count()
        .orderBy(col("count").desc())
        .first()
    )
    
    # Min and max values
    min_value = df.select(min(col(column))).first()[0]
    max_value = df.select(max(col(column))).first()[0]

    # Append results
    results.append({
        "column": column,
        "unique_values": unique_count,
        "most_frequent_value": top_value[0],
        "appearance_count": top_value[1],
        "min_value": min_value,
        "max_value": max_value
    })

# Create final DataFrame with ordered columns
results = spark.createDataFrame(results).select(
    "column",
    "unique_values",
    "most_frequent_value",
    "appearance_count",
    "min_value",
    "max_value"
)

# Display nicely in Databricks
display(results)

In [0]:
display(
    df.filter(col("phone") == col("emergency_contact"))
)

In [0]:
#from pyspark.sql.functions import length, col

df.select(
    "phone",
    length(col("phone")).alias("phone_length")
).show()

In [0]:
display(
    df.filter(length(col("phone")) != 10)
)

In [0]:
display(
    df.filter(
        ~substring(col("phone"), 1, 2).isin("11", "10", "15", "12")
    )
)

In [0]:
dup_phones = df.groupBy("phone") \
    .count() \
    .filter(col("count") > 1) \
    .select("phone")

display(df.join(dup_phones, "phone", "inner"))

In [0]:
dup_phones = df.groupBy("gender") \
    .count() \
    .filter(col("count") > 1) \
    .show()

In [0]:
columns_to_check = ["gender", "city","governorate", "blood_type"]

results = []

for c in columns_to_check:
    
    dup = df.groupBy(c) \
        .count() \
        .filter(col("count") > 1) \
        .withColumn("column", lit(c)) \
        .select("column", col(c).alias("value"), "count")
    
    results.append(dup)

final_df = results[0]
for r in results[1:]:
    final_df = final_df.union(r)

display(final_df)

## Data Cleaning

1- nullable = True
2- ID = Decimal
3- Phone and emerhency_contact has null
4- Full Name
5- Check phone and emergency contanct are not the same
6- Age
7- Handling error not exist like (duplicated id, duplicated record, null values,Male & Female)

**Invalid rows count** = number of records that fail your data quality rules.

---

## Simple meaning

Rows that are **wrong or unusable** for your system.

---

## Example (Patient table)

A row is invalid if:

### 1. Missing required data

* `patient_id = NULL`
* `national_id = NULL`

---

### 2. Wrong logic

* `birth_date > today` (future birth date)
* `age < 0`

---

### 3. Business rule violation

* `gender not in (Male, Female)`
* `blood_type invalid`
* `phone = emergency_contact`

---

### 4. Format issues

* phone too short/long
* national_id not numeric (if required)

---

## So:

```text id="v1"
invalid_rows_count = number of rows failing ANY rule
```

---

## Example

If you have 1000 patients:

* 950 valid
* 50 invalid

Then:

```text id="v2"
invalid_rows_count = 50
```

---

## Why it matters

Used for:

* data quality KPI
* auditing
* fixing source system issues
* deciding reject vs clean pipeline

---

If you want, I can help you design a **formal data quality rules table for your whole patient dataset**.


### Silver Transformation

In [0]:
# Rename column
df = rename_columns(df, {"phone": "patient_phone"})

# Create patient_name
df = df.withColumn(
    "patient_name",
    concat_ws(" ", col("first_name"), col("last_name"))
)

# Drop old columns
df = df.drop("first_name", "last_name")

In [0]:
from pyspark.sql.functions import year, current_date, when

df = (
    df.transform(lambda d: remove_duplicates(d, ["patient_id"]))
      .transform(trim_all_string_columns)
      .transform(replace_null_strings)
)

# Calculate age
df = df.withColumn("patient_age", year(current_date()) - year(col("birth_date")))

# concat 0 to phone numbers with length less than 10
df = df.withColumn("patient_phone", concat(lit("0"), col("patient_phone")))
df = format_egypt_phone_numbers(df, ["patient_phone"])

df = df.withColumn("emergency_contact", concat(lit("0"), col("emergency_contact")))
df = format_egypt_phone_numbers(df, ["emergency_contact"])

### write into silver table

In [0]:
display(df)

In [0]:
df.write \
  .mode("overwrite") \
  .option("overwriteSchema", "true") \
  .format("delta") \
  .saveAsTable("medical_insurance.silver.patient_silver")

In [0]:
%sql
select * from medical_insurance.silver.patient_silver